# Gulf of Tunis — Multi-Source Ocean Data Tutorial

This notebook shows how to access and visualize ocean observations over the **Gulf of Tunis** from two cloud-native data services:

| # | Variable | Sensor / Product | Service |
|---|----------|------------------|---------|
| 1 | Sea Surface Temperature (gridded, L3) | CMEMS SST MED NRT | Copernicus Marine |
| 2 | Sea water salinity (gridded, model) | CMEMS MED Physics MY 4.2 km | Copernicus Marine |
| 3 | Sea Surface Temperature (swath, L2) | Sentinel-3A SLSTR | Planetary Computer (STAC) |
| 4 | Chlorophyll-a (swath, L2) | Sentinel-3A OLCI | Planetary Computer (STAC) |

**Learning objectives:**
- Use the `copernicusmarine` Python client to stream gridded satellite and model data
- Plot basin-averaged time series and interactive spatial maps of a model variable (salinity)
- Use `pystac_client` to search a STAC API and open assets with `xarray`
- Clip, mask, and visualize 2-D swath data over a small area of interest
- Compare SST from two independent sources over the same region

## 0. Shared setup

We define the **bounding box** once and reuse it throughout. All three sections clip their data to this box.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import hvplot.xarray
import hvplot.pandas

# Area of interest: Gulf of Tunis  [lon_min, lat_min, lon_max, lat_max]
bbox = [10.0, 36.6, 10.9, 37.2]
print(f"Gulf of Tunis bbox: lon [{bbox[0]}, {bbox[2]}]  lat [{bbox[1]}, {bbox[3]}]")

---
## 1. CMEMS gridded SST (Copernicus Marine Service)

The **Copernicus Marine Service** (CMEMS) distributes processed, gridded satellite products via an OPeNDAP/Zarr backend. The `copernicusmarine` Python client lets us stream only the data we need — no download required.

We use:
- **Dataset:** `SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b`
- **Variable:** `adjusted_sea_surface_temperature` (Kelvin → °C)
- **Grid:** ~0.01° regular lat/lon over the Mediterranean

> **Credentials:** the `~/.netrc` file must contain your Copernicus Marine username and password.

In [ ]:
import copernicusmarine

dataset_id = "SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b"

ds_cmems = copernicusmarine.open_dataset(dataset_id)
print(list(ds_cmems.data_vars))

Select a single time step and clip to the Gulf of Tunis bbox using `.sel()`.

> If the latitude slice returns empty data, the axis may be stored in descending order — swap the limits: `slice(bbox[3], bbox[1])`.

In [ ]:
time_val="2026-01-01 16:00"
da_sst_cmems = (
    ds_cmems["adjusted_sea_surface_temperature"]
    .sel(time=time_val, method="nearest")
    .sel(
        longitude=slice(bbox[0], bbox[2]),
        latitude=slice(bbox[1], bbox[3]),
    )
    .load()
) - 273.15  # Kelvin → Celsius

print(f"Shape: {da_sst_cmems.shape}")
print(f"SST range: {float(da_sst_cmems.min()):.1f} – {float(da_sst_cmems.max()):.1f} °C")

In [ ]:
da_sst_cmems.hvplot(
    x="longitude", y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdYlBu_r",
    clabel="SST (°C)",
    #title=f"CMEMS Gridded SST — Gulf of Tunis {time_val}",
    tiles="OSM",
    frame_width=700,
)

---
## 2. CMEMS gridded salinity (Copernicus Marine Service)

To complement the SST field we add **sea water salinity** (practical salinity units, PSU) from the CMEMS Mediterranean Physics multi-year (MY) model product.

We use:
- **Dataset:** `cmems_mod_med_phy-sal_my_4.2km_P1D-m`
- **Variable:** `so` — sea water salinity (PSU)
- **Grid:** ~4.2 km regular lat/lon, 141 depth levels, daily timestep
- **Period loaded:** 1 January – 31 March 2026

The `copernicusmarine.open_dataset()` call accepts optional spatial and temporal bounding arguments, so only the Gulf of Tunis sub-region is streamed rather than the full Mediterranean domain.

> **Model vs. satellite:** unlike the SST above (a direct satellite observation), salinity at this resolution comes from a numerical ocean model constrained by observations. It is spatially and temporally gap-free but carries model uncertainty rather than a direct measurement error.

In [ ]:
import copernicusmarine

dataset_id = "cmems_mod_med_phy-sal_my_4.2km_P1D-m"

ds_sal = copernicusmarine.open_dataset(dataset_id,
    variables=["so"],
    minimum_longitude=10.0,
    maximum_longitude=10.9,
    minimum_latitude=36.6,
    maximum_latitude=37.2,
    start_datetime="2026-01-01T00:00:00",
    end_datetime="2026-03-31T00:00:00",
)
ds_sal

print(list(ds_sal.data_vars))

The dataset has 141 depth levels (from ~1 m down to ~5 600 m) and 90 daily timesteps. Let's inspect the structure before extracting data.

In [ ]:
print("Variables :", list(ds_sal.data_vars))
print("Dimensions:", dict(ds_sal.sizes))
print("Time range:", str(ds_sal.time.values[0])[:10], "->", str(ds_sal.time.values[-1])[:10])
#print("Depth levels:", ds_sal.depth.values)

### 2a. Load the surface layer

We select `depth=0` (the shallowest level, ~1 m) and call `.load()` to pull the full 90-day, 15×22 sub-region into memory — a small array (~30 KB), so this is near-instantaneous.

In [ ]:
%%time
so = ds_sal["so"].isel(depth=0).load()

print(f"Salinity min : {float(so.min()):.2f} PSU")
print(f"Salinity max : {float(so.max()):.2f} PSU")
print(f"Salinity mean: {float(so.mean()):.2f} PSU")

### 2b. Basin-averaged salinity time series

We collapse latitude and longitude to get a single daily mean, then plot it as a line chart. This shows the seasonal signal and any episodic freshening events — for example, Medjerda River flood pulses can lower surface salinity measurably in winter months.

In [ ]:
ts_sal = so.mean(dim=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(12, 4))
ts_sal.plot(ax=ax, color="steelblue", linewidth=1.5)
ax.set_ylabel("Salinity (PSU)")
ax.set_xlabel("Date")
ax.set_title("Basin-averaged daily surface salinity — Gulf of Tunisia, Jan–Mar 2026")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 2c. Interactive spatial map with time slider

`hvplot` with `groupby="time"` generates an interactive map with a **time slider** so we can step through individual days and watch salinity patterns evolve. `rasterize=True` keeps the display responsive even if the grid is large.

In [ ]:
import hvplot.xarray  # noqa: F401
import panel as pn
pn.extension()

so.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    rasterize=True,
    geo=True,
    tiles="OSM",
    cmap="cividis",
    clabel="PSU",
    clim=(float(so.min()), float(so.max())),
    title="Surface salinity (so, 1 m) — Gulf of Tunisia",
    frame_width=700,
)

In [ ]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search_sst = catalog.search(
    collections=["sentinel-3-slstr-wst-l2-netcdf"],
    bbox=bbox,
    datetime="2026-04-01/2026-04-21",
)

items_sst = search_sst.item_collection()
print(f"Found {len(items_sst)} Sentinel-3 SLSTR passes")
print(f"First pass: {items_sst[0].datetime}")

---
## 3. Sentinel-3 SLSTR SST — swath data via Planetary Computer (STAC)

As a cross-check against the CMEMS gridded SST we fetch a **Level-2 swath pass** from Sentinel-3 SLSTR (Sea and Land Surface Temperature Radiometer).

- **Collection:** `sentinel-3-slstr-wst-l2-netcdf`
- **Variable:** `sea_surface_temperature` (K → °C)
- **Resolution:** ~1 km along-track, ~1 km across-track

Data are accessed through the **Planetary Computer STAC API**, which indexes Copernicus Sentinel products on Azure. The `planetary_computer.sign_inplace` modifier automatically injects short-lived SAS tokens into each asset URL so no explicit authentication is required.

In [ ]:
item_sst = items_sst[0]

asset = planetary_computer.sign(item_sst.assets["l2p"])
ds_sst = xr.open_dataset(fsspec.open(asset.href).open(), decode_timedelta=False)

print(f"Dimensions: {dict(ds_sst.dims)}")
print(f"Coordinates: {list(ds_sst.coords)}")

The L2P file has a `nj × ni` swath grid (along-track × across-track) with 2-D `lat`/`lon` coordinate arrays — there is no regular grid, so we cannot use `.sel()`. Instead we build a boolean mask and extract pixels manually.

Each SLSTR L2P file covers one satellite pass with its own irregular `nj × ni` swath grid. We open the first matching item, apply a boolean lat/lon mask to clip to the Gulf of Tunis bbox, then extract only the rectangular envelope of the matching pixels. Pixels outside the bbox but inside that envelope are set to `NaN` so they don't appear in the plot.

In [ ]:
lat_sst = ds_sst["lat"].values
lon_sst = ds_sst["lon"].values

mask_sst = (
    (lat_sst >= bbox[1]) & (lat_sst <= bbox[3]) &
    (lon_sst >= bbox[0]) & (lon_sst <= bbox[2])
)

nj_idx, ni_idx = np.where(mask_sst)
print(f"{len(nj_idx)} pixels found in the Gulf of Tunis")

nj_min, nj_max = nj_idx.min(), nj_idx.max() + 1
ni_min, ni_max = ni_idx.min(), ni_idx.max() + 1

sst_sub = ds_sst["sea_surface_temperature"].isel(
    nj=slice(nj_min, nj_max), ni=slice(ni_min, ni_max), time=0
).load() - 273.15

lat_sub = lat_sst[nj_min:nj_max, ni_min:ni_max]
lon_sub = lon_sst[nj_min:nj_max, ni_min:ni_max]

# Apply exact bbox mask (remove rectangular envelope pixels outside bbox)
mask_sub = mask_sst[nj_min:nj_max, ni_min:ni_max]
sst_vals = sst_sub.values.copy()
sst_vals[~mask_sub] = np.nan

valid = sst_vals[np.isfinite(sst_vals) & (sst_vals > 0)]
print(f"SST range: {valid.min():.1f} – {valid.max():.1f} °C  |  mean: {valid.mean():.1f} °C")

In [ ]:
df_sst = pd.DataFrame({
    "lat": lat_sub.flatten(),
    "lon": lon_sub.flatten(),
    "sst": sst_vals.flatten(),
})
df_sst = df_sst[(df_sst["sst"] > 0) & (df_sst["sst"] < 40)]

df_sst.hvplot.points(
    x="lon", y="lat",
    c="sst",
    colormap="RdYlBu_r",
    clabel="SST (°C)",
    title=f"Sentinel-3 SLSTR SST — Gulf of Tunis ({str(item_sst.datetime.date())})",
    xlabel="Longitude", ylabel="Latitude",
    size=6,
    geo=True,
    tiles="OSM",
    frame_width=700,
)

---
## 4. Sentinel-3 OLCI Chlorophyll-a — swath data via Planetary Computer (STAC)

Chlorophyll-a (CHL) is a proxy for **phytoplankton biomass** — a key biological indicator of ocean productivity. We use the **Neural Net** algorithm output (`CHL_NN`) from the OLCI Water Full Resolution (WFR) product.

- **Collection:** `sentinel-3-olci-wfr-l2-netcdf`
- **Variable:** `CHL_NN` (mg/m³) — best for coastal and turbid waters
- **Resolution:** ~300 m

**Important:** the `chl_nn.nc` asset contains only the data values. Lat/lon coordinates live in a separate `geo-coordinates` asset and must be loaded and merged.

In [ ]:
search_chl = catalog.search(
    collections=["sentinel-3-olci-wfr-l2-netcdf"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-21",
)

items_chl = search_chl.item_collection()
print(f"Found {len(items_chl)} Sentinel-3 OLCI passes")
print(f"First pass: {items_chl[0].datetime}")

In [ ]:
def open_asset(item, key):
    """Sign and open a Planetary Computer asset as an xarray Dataset."""
    asset = planetary_computer.sign(item.assets[key])
    return xr.open_dataset(fsspec.open(asset.href).open())

item_chl = items_chl[0]

ds_chl = open_asset(item_chl, "chl-nn")
geo    = open_asset(item_chl, "geo-coordinates")

# Merge lat/lon (stored in a separate file) as coordinates
ds_chl = ds_chl.assign_coords(
    latitude=(["rows", "columns"], geo["latitude"].values),
    longitude=(["rows", "columns"], geo["longitude"].values),
)

print(ds_chl)

Same spatial masking approach as for SST: boolean mask → rectangular slice → re-apply mask.

In [ ]:
lat_chl = ds_chl["latitude"].values
lon_chl = ds_chl["longitude"].values

mask_chl = (
    (lat_chl >= bbox[1]) & (lat_chl <= bbox[3]) &
    (lon_chl >= bbox[0]) & (lon_chl <= bbox[2])
)

rows_idx, cols_idx = np.where(mask_chl)
print(f"{len(rows_idx)} pixels found in the Gulf of Tunis")

row_min, row_max = rows_idx.min(), rows_idx.max() + 1
col_min, col_max = cols_idx.min(), cols_idx.max() + 1

chl_sub = ds_chl["CHL_NN"].isel(
    rows=slice(row_min, row_max),
    columns=slice(col_min, col_max),
).load()

lat_sub_chl = lat_chl[row_min:row_max, col_min:col_max]
lon_sub_chl = lon_chl[row_min:row_max, col_min:col_max]

mask_sub_chl = mask_chl[row_min:row_max, col_min:col_max]
chl_vals = chl_sub.values.copy()
chl_vals[~mask_sub_chl] = np.nan

valid_chl = chl_vals[np.isfinite(chl_vals) & (chl_vals > 0)]
print(f"CHL_NN range: {valid_chl.min():.3f} – {valid_chl.max():.3f} mg/m³  |  mean: {valid_chl.mean():.3f} mg/m³")

In [ ]:
df_chl = pd.DataFrame({
    "lat": lat_sub_chl.flatten(),
    "lon": lon_sub_chl.flatten(),
    "chl": chl_vals.flatten(),
})
df_chl = df_chl[df_chl["chl"].notna() & (df_chl["chl"] > 0) & (df_chl["chl"] < 100)]
df_chl["log_chl"] = np.log10(df_chl["chl"])

df_chl.hvplot.points(
    x="lon", y="lat",
    c="log_chl",
    colormap="YlGn",
    clim=(df_chl["log_chl"].quantile(0.02), df_chl["log_chl"].quantile(0.98)),
    clabel="log₁₀ CHL (mg/m³)",
    title=f"Sentinel-3 OLCI Chlorophyll-a — Gulf of Tunis ({str(item_chl.datetime.date())})",
    xlabel="Longitude", ylabel="Latitude",
    size=6,
    geo=True,
    tiles="OSM",
    frame_width=700,
)

---
## 5. Forecast precipitation with ICON Europe

In [ ]:
import os
import requests
import time
from datetime import datetime, timezone
from tqdm import tqdm

# Today's run date at 00h UTC
today = datetime.now(timezone.utc)
run_date = today.strftime("%Y%m%d00")

# Parameters
base_url = "https://opendata.dwd.de/weather/nwp/icon-eu/grib/00/tot_prec/"
output_dir = "../DATA/grib_files"
os.makedirs(output_dir, exist_ok=True)

# Forecast steps to download
xx_values = list(range(0, 79)) + list(range(81, 121, 3))

# Skip entirely if all files for today are already present
expected_files = [
    os.path.join(output_dir, f"icon-eu_europe_regular-lat-lon_single-level_{run_date}_{xx:03d}_TOT_PREC.grib2.bz2")
    for xx in xx_values
]
if all(os.path.exists(f) for f in expected_files):
    print(f"All TOT_PREC files for {run_date} already downloaded. Skipping.")
else:
    def download_file(xx):
        filename = f"icon-eu_europe_regular-lat-lon_single-level_{run_date}_{xx:03d}_TOT_PREC.grib2.bz2"
        file_path = os.path.join(output_dir, filename)
        file_url = f"{base_url}{filename}"

        if os.path.exists(file_path):
            return

        try:
            response = requests.get(file_url, timeout=30)
            response.raise_for_status()
            with open(file_path, "wb") as f:
                f.write(response.content)
        except requests.exceptions.RequestException:
            pass

        time.sleep(1)

    for xx in tqdm(xx_values, desc="Downloading TOT_PREC files", unit="file"):
        download_file(xx)

---
## 6. Forecast temperature with ICON

In [ ]:
import os
import requests
import time
from datetime import datetime, timezone
from tqdm import tqdm

# Today's run date at 00h UTC
today = datetime.now(timezone.utc)
run_date = today.strftime("%Y%m%d00")

# Parameters
base_url = "https://opendata.dwd.de/weather/nwp/icon-eu/grib/00/t_2m/"
output_dir = "../DATA/grib_files"
os.makedirs(output_dir, exist_ok=True)

# Forecast steps to download
xx_values = list(range(0, 79)) + list(range(81, 121, 3))

# Skip entirely if all files for today are already present
expected_files = [
    os.path.join(output_dir, f"icon-eu_europe_regular-lat-lon_single-level_{run_date}_{xx:03d}_T_2M.grib2.bz2")
    for xx in xx_values
]
if all(os.path.exists(f) for f in expected_files):
    print(f"All T_2M files for {run_date} already downloaded. Skipping.")
else:
    def download_file(xx):
        filename = f"icon-eu_europe_regular-lat-lon_single-level_{run_date}_{xx:03d}_T_2M.grib2.bz2"
        file_path = os.path.join(output_dir, filename)
        file_url = f"{base_url}{filename}"

        if os.path.exists(file_path):
            return

        try:
            response = requests.get(file_url, timeout=30)
            response.raise_for_status()
            with open(file_path, "wb") as f:
                f.write(response.content)
        except requests.exceptions.RequestException:
            pass

        time.sleep(1)

    for xx in tqdm(xx_values, desc="Downloading T_2M files", unit="file"):
        download_file(xx)

---
## 7. Interactive forecast maps — Gulf of Tunis

We read all downloaded GRIB2 files (both `TOT_PREC` and `T_2M`), clip them to the Gulf of Tunis bounding box, and assemble two `xarray.DataArray` objects with a `time` dimension. A **Panel / HoloViews** widget then lets you scrub through the forecast horizon with a time slider.

The colorbar is displayed as a **2-D color square** (`colorbar_opts={"major_label_text_font_size": ...}` with `colorbar=True` and square aspect via `color_levels`) rather than the default thin bar.

---
## 8. Summary

| Variable | Source | Type | Resolution | Period |
|----------|--------|------|------------|--------|
| SST (gridded) | CMEMS L3S NRT | Satellite composite | ~1 km | Jan 2026 |
| Salinity | CMEMS MED Physics MY | Ocean model | ~4.2 km | Jan–Mar 2026 |
| SST (swath) | Sentinel-3A SLSTR L2P | Swath (irregular) | ~1 km | Apr 2026 |
| Chlorophyll-a | Sentinel-3A OLCI WFR L2 | Swath (irregular) | ~300 m | varies |
| 2 m temperature | ICON-EU NWP | Gridded NWP forecast | ~7 km | 0–120 h |
| Total precipitation | ICON-EU NWP | Gridded NWP forecast | ~7 km | 0–120 h |

**Key points:**
- The CMEMS SST product is a gridded, gap-filled composite — convenient for time series and spatial comparisons, but spatially smoothed.
- The CMEMS salinity product is model-derived — gap-free and available at all depths, but carries model uncertainty rather than a direct observation error.
- The Planetary Computer STAC products are raw Level-2 swaths — higher spatial resolution but cloudy pixels are missing and the satellite only passes every ~2 days.
- Chlorophyll-a is displayed on a **log₁₀ scale** because its natural variability spans several orders of magnitude (0.01 – 10 mg/m³ in coastal seas).
- ICON-EU GRIB2 files use simple packing with sign-magnitude scale factors — parsed here without `cfgrib`/`eccodes` by reading section offsets directly.
- The Gulf of Tunis is a semi-enclosed, relatively shallow sea influenced by the Medjerda River discharge in the south — this drives elevated turbidity and chlorophyll near the coast, and can freshen surface salinity after heavy rainfall.

**Next steps:**
- Add in-situ profile data from the Copernicus Marine in-situ service to validate satellite SST and model salinity
- Extend the chlorophyll and SST swath searches to multiple passes and build time series
- Overlay surface currents from a CMEMS physics model to explain spatial SST and salinity patterns
- Compute temperature-salinity (T-S) diagrams to identify water masses in the Gulf